#II. Bài tập hướng dẫn mẫu
a.Thuật giải AKT vào bài toán 8 puzzle(n=3)

In [36]:
import copy
from heapq import heappush, heappop

n = 3

rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class PriorityQueue:
    def __init__(self):
        self.heap = []

    def push(self, item):
        heappush(self.heap, item)

    def pop(self):
        return heappop(self.heap)

    def empty(self):
        return len(self.heap) == 0

class Node:
    def __init__(self, parent, mats, empty_tile_posi, cost, level):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.cost = cost
        self.level = level

    def __lt__(self, other):
        return self.cost < other.cost

    def calculate_cost(self, final):
        count = 0
        for i in range(n):
            for j in range(n):
                if self.mats[i][j] != 0 and self.mats[i][j] != final[i][j]:
                    count += 1
        return count


def new_node(mats, empty_pos, new_pos, level, parent, final):
    new_mats = copy.deepcopy(mats)

    x1, y1 = empty_pos
    x2, y2 = new_pos

    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    node = Node(parent, new_mats, new_pos, 0, level)
    node.cost = node.calculate_cost(final)

    return node


def is_safe(x, y):
    return 0 <= x < n and 0 <= y < n


def print_matrix(mat):
    for row in mat:
        print(*row)
    print()


def print_path(node):
    if node is None:
        return
    print_path(node.parent)
    print_matrix(node.mats)


def solve(initial, empty_pos, final):
    pq = PriorityQueue()

    root = Node(None, initial, empty_pos, 0, 0)
    root.cost = root.calculate_cost(final)

    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()

        if minimum.cost == 0:
            print_path(minimum)
            return

        for i in range(4):
            new_pos = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i]
            ]

            if is_safe(new_pos[0], new_pos[1]):
                child = new_node(
                    minimum.mats,
                    minimum.empty_tile_posi,
                    new_pos,
                    minimum.level + 1,
                    minimum,
                    final
                )
                pq.push(child)

initial = [
    [1, 2, 3],
    [5, 6, 0],
    [7, 8, 4]
]

final = [
    [1, 2, 3],
    [5, 8, 6],
    [0, 7, 4]
]

empty_tile_posi = [1, 2]

solve(initial, empty_tile_posi, final)

1 2 3
5 6 0
7 8 4

1 2 3
5 0 6
7 8 4

1 2 3
5 8 6
7 0 4

1 2 3
5 8 6
0 7 4



# III. Bài tập ở lớp
Bài 1. Cài đặt thuật giải AKT vào bài toán 15 puzzle(n=4).

In [37]:
import heapq
import copy

class PuzzleState:
    def __init__(self, board, parent=None, move="", g=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.g = g
        self.n = len(board)
        self.h = self.calculate_manhattan()
        self.f = self.g + self.h

    def calculate_manhattan(self):
        distance = 0
        for i in range(self.n):
            for j in range(self.n):
                val = self.board[i][j]
                if val != 0:
                    target_x = (val - 1) // self.n
                    target_y = (val - 1) % self.n
                    distance += abs(i - target_x) + abs(j - target_y)
        return distance

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for i in range(self.n):
            for j in range(self.n):
                if self.board[i][j] == 0:
                    return i, j

    def generate_neighbors(self):
        neighbors = []
        x, y = self.get_blank_pos()
        directions = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

        for dx, dy, move_name in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.n and 0 <= ny < self.n:
                new_board = copy.deepcopy(self.board)
                new_board[x][y], new_board[nx][ny] = new_board[nx][ny], new_board[x][y]
                neighbors.append(PuzzleState(new_board, self, move_name, self.g + 1))
        return neighbors

def print_path(node):
    path = []
    while node:
        path.append(node)
        node = node.parent
    path.reverse()
    for step in path:
        print(f"Move: {step.move}")
        for row in step.board:
            print(row)
        print("f =", step.f, "| g =", step.g, "| h =", step.h)
        print("-" * 22)

def a_star_15_puzzle(start_board):
    start_node = PuzzleState(start_board)
    open_list = []
    heapq.heappush(open_list, start_node)
    closed_set = set()

    while open_list:
        current_node = heapq.heappop(open_list)

        if current_node.h == 0:
            print("Đã tìm thấy giải pháp!")
            print_path(current_node)
            return

        board_tuple = tuple(tuple(row) for row in current_node.board)
        if board_tuple in closed_set:
            continue
        closed_set.add(board_tuple)

        for neighbor in current_node.generate_neighbors():
            neighbor_tuple = tuple(tuple(row) for row in neighbor.board)
            if neighbor_tuple not in closed_set:
                heapq.heappush(open_list, neighbor)

start = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 0, 11],
    [13, 14, 15, 12]
]

if __name__ == "__main__":
    print("Giải bài toán 15-puzzle (n=4) bằng A*:")
    a_star_15_puzzle(start)

Giải bài toán 15-puzzle (n=4) bằng A*:
Đã tìm thấy giải pháp!
Move: 
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 0, 11]
[13, 14, 15, 12]
f = 2 | g = 0 | h = 2
----------------------
Move: Right
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 0]
[13, 14, 15, 12]
f = 2 | g = 1 | h = 1
----------------------
Move: Down
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]
f = 2 | g = 2 | h = 0
----------------------


# IV. Bài tập về nhà
Bài 1. Cài đặt thuật giải AKT vào bài toán 15 puzzle(n>4).

In [72]:
import heapq
import copy

dx = [1,-1,0,0]
dy = [0,0,1,-1]

def tao_goal(n):
    goal = []
    dem = 1
    for i in range(n):
        hang = []
        for j in range(n):
            if dem == n*n:
                hang.append(0)
            else:
                hang.append(dem)
            dem += 1
        goal.append(hang)
    return goal

def tinh_h(a, goal, n):
    dem = 0
    for i in range(n):
        for j in range(n):
            if a[i][j] != 0:
                val = a[i][j] - 1
                x = val // n
                y = val % n
                dem += abs(i - x) + abs(j - y)
    return dem

def tim_0(a, n):
    for i in range(n):
        for j in range(n):
            if a[i][j] == 0:
                return i, j
    return -1, -1

def in_ma_tran(a, n):
    for i in range(n):
        for j in range(n):
            if a[i][j] == 0:
                print("   ", end="")
            else:
                print(f"{a[i][j]:3}", end="")
        print()

def AKT(start, n):
    goal = tao_goal(n)

    OPEN = []
    CLOSE = set()

    heapq.heappush(OPEN, (tinh_h(start, goal, n), 0, start, []))

    while OPEN:
        f, g, N, path = heapq.heappop(OPEN)

        keyN = str(N)
        if keyN in CLOSE:
            continue

        if N == goal:
            return path + [N]

        CLOSE.add(keyN)

        x, y = tim_0(N, n)

        for k in range(4):
            nx = x + dx[k]
            ny = y + dy[k]

            if 0 <= nx < n and 0 <= ny < n:
                S = copy.deepcopy(N)
                S[x][y], S[nx][ny] = S[nx][ny], S[x][y]

                keyS = str(S)
                if keyS in CLOSE:
                    continue

                gS = g + 1
                hS = tinh_h(S, goal, n)

                heapq.heappush(OPEN, (gS + hS, gS, S, path + [N]))

    return None

def in_loi_giai(kq, n):
    goal = tao_goal(n)

    for i, buoc in enumerate(kq):
        g = i
        h = tinh_h(buoc, goal, n)
        f = g + h

        print(f"Buoc {i+1}:")
        print(f"g = {g}, h = {h}, f = {f}")
        in_ma_tran(buoc, n)
        print("----------------------")

n = 5

start = [
    [1,2,3,4,5],
    [6,7,8,9,10],
    [11,12,13,14,15],
    [16,17,18,0,20],
    [21,22,23,19,24]
]

kq = AKT(start, n)

if kq:
    in_loi_giai(kq, n)
    print("Đã giải xong!")
else:
    print("That bai")

Buoc 1:
g = 0, h = 2, f = 2
  1  2  3  4  5
  6  7  8  9 10
 11 12 13 14 15
 16 17 18    20
 21 22 23 19 24
----------------------
Buoc 2:
g = 1, h = 1, f = 2
  1  2  3  4  5
  6  7  8  9 10
 11 12 13 14 15
 16 17 18 19 20
 21 22 23    24
----------------------
Buoc 3:
g = 2, h = 0, f = 2
  1  2  3  4  5
  6  7  8  9 10
 11 12 13 14 15
 16 17 18 19 20
 21 22 23 24   
----------------------
Đã giải xong!
